# Clase 9: redes generativas antagónicas (GAN)

En esta clase construyes y entrenas, en directo, una red generativa antagónica (GAN, del inglés *Generative Adversarial Network*) mínima. Ejecutas cada celda de código y observas cómo el generador aprende a imitar una distribución de datos real.

## Objetivos de aprendizaje

Al terminar esta clase, sabes:

- Explicar la relación entre el generador y el discriminador con la analogía del falsificador y el policía
- Describir el objetivo minimax que entrena ambas redes a la vez
- Implementar en PyTorch un generador y un discriminador con capas lineales
- Entrenar una GAN mínima sobre datos sintéticos de dos dimensiones
- Visualizar el progreso del entrenamiento comparando datos reales y datos generados
- Reconocer los motivos que hacen inestable el entrenamiento de una GAN, entre ellos el colapso de modo

## El falsificador y el policía

Imagina a una persona falsificadora de billetes y a la policía que intenta detectarla. Al principio, los billetes falsos son toscos y la policía los detecta sin esfuerzo. La persona falsificadora aprende de cada rechazo y mejora el papel, la tinta, los detalles. La policía también aprende y afina su criterio. Con el tiempo, ambas partes mejoran en paralelo.

Esa dinámica es una GAN:

- El **generador** hace de falsificador o falsificadora: parte de ruido aleatorio y produce datos que intenta hacer pasar por reales
- El **discriminador** hace de policía: recibe datos reales y datos generados, y decide para cada uno si es auténtico o falso

Ninguna de las dos redes ve el objetivo final de la otra. El generador nunca observa datos reales: solo recibe la señal del discriminador sobre si sus creaciones cuelan o no. Cuanto mejor discrimina el discriminador, más información recibe el generador sobre en qué está fallando.

## El juego minimax

El generador y el discriminador entrenan a la vez, con objetivos opuestos:

- El discriminador quiere **maximizar** su acierto: alta probabilidad para los datos reales, baja probabilidad para los datos generados
- El generador quiere **minimizar** ese mismo acierto: quiere que el discriminador confunda sus creaciones con datos reales

En la práctica, entrenas las dos redes de forma alternada:

1. Actualizas el discriminador para que distinga mejor lo real de lo generado
2. Actualizas el generador para que engañe mejor al discriminador
3. Repites el ciclo miles de veces

El punto ideal de llegada es un equilibrio en el que el discriminador ya no acierta mejor que lanzando una moneda al aire: en ese momento, los datos generados son estadísticamente indistinguibles de los reales.

Un detalle práctico: en vez de minimizar directamente la probabilidad de ser detectado (una función con gradiente casi plano al principio del entrenamiento), el generador usa una variante que maximiza la probabilidad de que el discriminador lo clasifique como real. El objetivo apunta en la misma dirección, pero da un gradiente mucho más útil desde el primer paso.

## Construye una GAN mínima

Construyes ahora una GAN completa en PyTorch:

- **Datos reales**: puntos que forman un círculo (una mezcla de ocho grupos gaussianos), generados con NumPy
- **Generador**: dos capas ocultas; toma un vector de ruido en dos dimensiones y devuelve un punto en dos dimensiones
- **Discriminador**: dos capas ocultas; toma un punto en dos dimensiones y devuelve un logit de "realidad"

Ejecuta la celda siguiente para definir los datos, los dos modelos, la función de pérdida y los optimizadores.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Datos reales: mezcla de ocho gaussianas en círculo ---
def sample_real_data(n_samples, n_modes=8, radius=4.0, std=0.15):
    angles = 2 * np.pi * np.random.randint(0, n_modes, size=n_samples) / n_modes
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    points = centers + std * np.random.randn(n_samples, 2)
    return torch.tensor(points, dtype=torch.float32)


LATENT_DIM = 2


# --- Generador: ruido latente -> punto en dos dimensiones ---
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.net(z)


# --- Discriminador: punto en dos dimensiones -> logit de "realidad" ---
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),  # logit, sin sigmoide: se aplica dentro de la pérdida
        )

    def forward(self, x):
        return self.net(x)


generator = Generator().to(device)
discriminator = Discriminator().to(device)

# BCEWithLogitsLoss combina sigmoide y entropía cruzada binaria de forma numéricamente estable
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

print(generator)
print(discriminator)

## Observa el entrenamiento en directo

Ejecuta la celda siguiente y deja que corra. Cada cierto número de pasos, el código actualiza un diagrama de dispersión con los datos reales (los ocho grupos en círculo) y los datos que produce el generador en ese momento.

Fíjate en tres cosas mientras avanza:

- Al principio, los puntos generados forman una nube difusa sin relación con el círculo
- A medida que avanza el entrenamiento, la nube se acerca a los ocho grupos reales
- Comprueba, al final, si el generador cubre los ocho grupos o si se concentra solo en dos o tres: eso es el colapso de modo

In [ ]:
BATCH_SIZE = 256
N_STEPS = 4000
PLOT_EVERY = 500

for step in range(1, N_STEPS + 1):
    # ---------- 1) Actualiza el discriminador ----------
    real_data = sample_real_data(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z).detach()  # detach: el generador no se actualiza en este paso

    real_labels = torch.ones(BATCH_SIZE, 1, device=device)
    fake_labels = torch.zeros(BATCH_SIZE, 1, device=device)

    loss_d_real = criterion(discriminator(real_data), real_labels)
    loss_d_fake = criterion(discriminator(fake_data), fake_labels)
    loss_d = loss_d_real + loss_d_fake

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Actualiza el generador (pérdida no saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z)
    # El generador "quiere" que el discriminador clasifique sus muestras como reales
    loss_g = criterion(discriminator(fake_data), real_labels)

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    # ---------- 3) Dibuja el progreso cada PLOT_EVERY pasos ----------
    if step % PLOT_EVERY == 0 or step == 1:
        with torch.no_grad():
            real_vis = sample_real_data(500).numpy()
            z_vis = torch.randn(500, LATENT_DIM, device=device)
            fake_vis = generator(z_vis).cpu().numpy()

        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(real_vis[:, 0], real_vis[:, 1], s=10, alpha=0.5, label="Datos reales")
        ax.scatter(fake_vis[:, 0], fake_vis[:, 1], s=10, alpha=0.5, label="Datos generados")
        ax.set_title(f"Paso {step} de {N_STEPS} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")
        ax.legend(loc="upper right")
        ax.set_xlim(-6, 6)
        ax.set_ylim(-6, 6)
        ax.set_aspect("equal")
        plt.show()

## Por qué las GAN son difíciles de entrenar

Entrenar una GAN es más frágil que entrenar una red que solo clasifica o predice. Estos son los tres motivos principales:

- **Inestabilidad del juego minimax**. Las dos redes se actualizan a la vez persiguiendo objetivos opuestos. Si el discriminador mejora demasiado rápido, deja al generador sin gradiente útil; si mejora demasiado despacio, deja pasar datos de mala calidad. El sistema puede oscilar en lugar de asentarse en un equilibrio
- **Colapso de modo**. El generador descubre que unas pocas salidas concretas engañan bien al discriminador y deja de explorar el resto de la distribución real. El resultado es poca variedad en las muestras, aunque cada una por separado parezca convincente
- **Ausencia de una métrica de progreso fiable**. Al ser un juego entre dos jugadores que mejoran a la vez, el valor de la pérdida no decrece de forma monótona como en un problema de optimización clásico. No puedes leer directamente en la curva de pérdida si el modelo mejora o no

La raíz teórica de esta fragilidad está en la divergencia que mide el objetivo minimax clásico. Cuando la distribución de los datos generados no se solapa con la distribución real —algo muy habitual al principio del entrenamiento—, esa divergencia deja de dar una señal de aprendizaje útil. Avances como la red generativa antagónica de Wasserstein (WGAN) sustituyen esa divergencia por una distancia que sigue dando gradiente útil incluso sin solapamiento, lo que estabiliza el entrenamiento.

## Resumen de puntos clave

- Una GAN combina dos redes que compiten: el generador crea datos y el discriminador los juzga
- El objetivo de entrenamiento es un juego minimax: el discriminador maximiza su acierto y el generador lo minimiza
- En la práctica, el generador usa una variante no saturante que da gradientes más útiles al principio del entrenamiento
- Con capas lineales sencillas y datos sintéticos en dos dimensiones ya puedes observar el aprendizaje y el riesgo de colapso de modo
- La inestabilidad del entrenamiento es estructural, no un error de implementación: viene de las propiedades de la divergencia que minimiza el juego minimax

**Abre ahora el notebook de práctica companion `../notebooks_practica/clase09_practica_gan.ipynb` y experimenta con los parámetros del entrenamiento.**